# Methods to handle non-detections (censored data) 

This tutorial will guide you through a few of the basic methods to make use of non-detections in astronomical data. Specifically:

1. Obtaining mean values through stacking individual non-detections
2. Obtaining information about sample distributions using the Kaplan-Meier estimator
3. Testing for differences between distributions using the log rank test.

## 0. Load relevant packages

You'll need the following packages in order to run this notebook:
- numpy
- matplotlib
- astropy
- MangaHIStacking (included in repo)
- lifelines

Use "pip install [package name]" to install any that you don't currently have.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.io import fits, ascii
from MangaHIStacking import stack_control
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

## 1. Stacking

First we will use a code to "stack" data. The data come from the HI-MaNGA survey, a program to observe 21cm spectral line emission in all galaxies from the SDSS MaNGA survey. 21cm emission is a direct tracer of neutral atomic hydrogen gas (HI). 

HI-MaNGA has a fixed integration time, meaning we are sensitive down to a fixed apparent brightness of the 21cm spectral line. Because some galaxies contain low quantities of HI, and some galaxies are further away, we do not detect everything we observe, but we do obtain upper limits on the 21cm brightness.

Stacking is a process of averaging together many observations of similar objects in order to obtain a detection, even if nothing was detected in the individual observations. This brightness of the detection represents the mean of the sample being stacked. This is all possible because the noise level of our stacked image goes down as 1/sqrt(N), assuming the noise levels of each image/spectra are about the same. 

Stacking HI-MaNGA spectra involves first aligning each spectram so the expected galaxy emission is at the center. Then the aligned spectra are averaged. We provide the code that does these steps, but you'll set up the sample to be stacked.

### Load the catalog

First, load catalog. We've combined the MaNGA-HI, MaNGA DRP, and MaNGA DAP catalogs for you. Data models for all the different columns can be found at:
- HI-MaNGA: https://data.sdss.org/datamodel/files/MANGA_HI/HIPVER/mangaHIall.html
- DRP: https://data.sdss.org/datamodel/files/MANGA_SPECTRO_REDUX/DRPVER/drpall.html
- DAP: https://data.sdss.org/datamodel/files/MANGA_SPECTRO_ANALYSIS/DRPVER/DAPVER/dapall.html

In [ ]:
# load the catalog (replace this with Karen's copy)
cat = Table.read('himanga_drpall_dapall.fits')

# for simplicity, I'm going to filter out the ALFALFA data. Additional cuts are made on data quality
sel = (cat['SESSION'] != 'ALFALFA') & (cat['BLSTRUCT'] == 0) & (cat['NEGDET'] == 0)
cat = cat[sel]

# we have some duplicate entries in our catalog. Let's just exctact the unique entries.
unique_names, unique_inds = np.unique(cat['MANGAID'], return_index=True)
cat = cat[unique_inds]
cat

Next, let's define the subsample over which we want to stack. For this demonstration, let's select all non-detections (LOGHILIMS00KMS > 0) with stellar masses between 10^9 and 10^9.5 Msun. Let's plot the first one to confirm it's a non-detetion.

In [ ]:
# selector
sel =  (cat['LOGMSTARS'] >= 9) & (cat['LOGMSTARS'] < 9.5) & (cat['LOGHILIM200KMS'] > 0)

# create a new table with just this subsample
subsample = cat[sel]

# just to prove that this is indeed a non-detection, let's plot the spectrum:
spec_path = './spectra/gbt/csv/mangahi-' + subsample['PLATEIFU'][0] + '.csv' # define the path to the spectrum
spec=ascii.read(spec_path, data_start=20, names=['vhi', 'fhi', 'bhi'])

# plot the spectrum
fig, ax = plt.subplots()
ax.plot(spec['vhi'].squeeze(), spec['fhi'].squeeze())
ax.set_xlim(subsample['VOPT'][0]-1000, subsample['VOPT'][0]+1000)
ax.set_xlabel('recession velocity [km/s]')
ax.set_ylabel('flux density [Jy]')
ax.set_title(subsample['PLATEIFU'][0])

Now let's run the stacking code. It requires an input table in a very specific format. The first few lines of code below make that table.

The stacking routine also requires an output spectrum width as well as a normalization method. The choices for the normalization are "stellarmass" and "distance". The first one converts the spectrum into HI-to-stellar mass ratio by scaling by distance^2 and dividing by stellar mass. The second converts it into HI mass by just scaling by distance^2. The point of this normalization is to put everything in the same absolute scale. We'll use "stellarmass" normalization.

***Note: the code may tell you it can't find a number of spectra. Ignore that for now. There should be plentry of spectra still available. This is something I'm looking into.***

In [ ]:
# these lines just set up the input table for the stacking code
tbl = Table()
new_colnames = ['plateifu', 'mangaid', 'logmstars', 'sini', 'vhi', 'rms']
old_colnames = ['PLATEIFU', 'MANGAID', 'LOGMSTARS', 'SINI', 'VOPT', 'RMS']

for new, old in zip(new_colnames, old_colnames):
    tbl[new] = subsample[old]

# define the desired spectrum width
spectrum_width = 500 # km/s

# run the stacking code
x_val, fHI = stack_control(tbl, width=spectrum_width, method='stellarmass')

# plot the resulting spectrum
fig, ax = plt.subplots()
ax.plot(x_val, fHI)
ax.set_xlabel('velocity [km/s]')
ax.set_ylabel('gas-to-stellar mass ratio spectrum')
ax.set_title('Stacked spectrum')

Hey look, a detection! There you have it: the average signal from the few hundred undetected galaxies with M*=10^9-10^9.5! 

You may notice some residual structure in the baseline level. This occurs when some spectra have not been baseline subtracted across the full spectral window. Such structure can be removed.

### ***Your turn***

Come up with a few selections of your own. You can look at different stellar mass ranges, color regimes, etc. Look through the available parameters in the DAPALL and DRPALL files. Display the spectra on the same plot.

In [ ]:
# Insert your code here. Make additional blocks as needed

## 2. Determining cumulative probability distributions with lifelines

The Kaplan-Meier estimator is a useful tool to estimate probability distributions in the presence of censored data. It estimates the "survival function", S, whih is the probability that an "event" has not occurred by a given time, t. We can translate this idea to brightnesses by thinking of the survival function as the probability that an object has not been detected by a given brightness level. 

Once we have the survival function, we can get the cumulative distribution function, F,  by the fact that F = 1 - S.

We'll demonstrate how to use the "lifelines" package to get all this information (https://lifelines.readthedocs.io/en/latest/index.html)

First, we're going to create a cumulative probability distribution for HI-to-stellar mass ratio using lifelines KaplanMeierFitter class. We have lots of undetected 21cm observations, making Kaplan-Meier an appropriate approach to this problem. 

Keep in mind that Kaplan-Meier is designed for use with "right-censored" data, i.e, lower limits, whereas our 21cm non-detections are upper limits.  Normally, one would have to invert their data to make all the upper limits into lower limits, run the Kaplan-Meier estimator, then convert back. Thankfully, lifelines does that work for us.

We'll start by loading our catalog and defining the sample. Again for simplicity, we remove ALFALFA data and select unique entries.

In [ ]:
# load the catalog (replace this with Karen's copy)
cat = Table.read('himanga_drpall_dapall.fits')

# filter out the ALFALFA data
sel = (cat['SESSION'] != 'ALFALFA') 
cat = cat[sel]

# get unique entries
unique_names, unique_inds = np.unique(cat['mangaid'], return_index=True)
cat = cat[unique_inds]
cat

The catalog splits up detections and non-detections, but we need to put these in a single array, as well as create a second array saying which rows are the upper limits

In [ ]:
# create HI-to-stellar mass ratio column, called "gs", first with the detections (non-detections are LOGMHI = -999)
cat['gs'] = cat['LOGMHI'] - cat['LOGMSTARS']

# flag the ones that are upper limits and add the upper limits to the 'gs' column
limit = cat['LOGHILIM200KMS'] > 0
cat['gs'][limit] = cat['LOGHILIM200KMS'][limit] - cat['LOGMSTARS'][limit]

# save the upper limit array in the table too
cat['limit'] = limit

# I'm applying another selection to avoid exessively bad data
sel = (cat['gs'] > -99) & (cat['gs'] < 99) & (cat['LOGMSTARS'] > 8)
cat = cat[sel]

Let's make a quick scatter plot showing HI-to-stellar mass ratio vs. stellar mass to see how the detections and upper limits distribute themselves. We'll also print the number of detections and non-detections

In [ ]:
fig, ax = plt.subplots()

# plot detections
ax.scatter(cat['LOGMSTARS'][~cat['limit']], cat['gs'][~cat['limit']],s=5,alpha=0.5, label='detections')

# plot upper limits
ax.scatter(cat['LOGMSTARS'][cat['limit']], cat['gs'][cat['limit']],s=3, alpha=0.5, marker='v', label='non-detections')

ax.set_xlabel('log stellar mass [Msun]')
ax.set_ylabel('log HI-to-stellar mass ratio')

ax.legend()

print('number of detections: ', np.sum(~cat['limit']))
print('number of non-detections: ', np.sum(cat['limit']))


Next, we initiate the KaplanMeierFitter class. We supply it with an array of observations ("events"), and which of those observations are "observed" (detected). We use the fit_left_censoring() function since our data are left censored.

For comparison, let's do the same thing but with just the detections to see how incorporating the non-detections changes the distribution.

In [ ]:
# initiate KaplanMeierFitter class.
kmf = KaplanMeierFitter()
kmf.fit_left_censoring(durations=cat['gs'], event_observed=~cat['limit'])

# Do again, but limit to just detections (KM just gives a standard distribution function in this case)
kmf_dets = KaplanMeierFitter()
kmf_dets.fit_left_censoring(durations=cat['gs'][~cat['limit']], event_observed=~cat['limit'][~cat['limit']])

Below we plot the cumulative distribution functions and the median survival time (i.e., the middle of the distribution)

In [ ]:
# let's plot the results
fig, ax = plt.subplots()

# the version with upper limits
kmf.plot_cumulative_density(ax=ax, label='CDF with limits')

# the version with just detections
kmf_dets.plot_cumulative_density(ax=ax, label='CDF with only detections')

ax.set_xlabel('HI-to-stellar mass ratio')
ax.set_ylabel('F')

# print middle of distributions
print('mean value of full distribution: ', kmf.median_survival_time_)
print('mean value of just detections: ', kmf_dets.median_survival_time_)



Note how the curve with detections only is significantly higher than the curve with the upper limits includes. Clearly ignoring upper limits is not a good idea if you want an unbiased view of a population.

We can also  the survival function, which for this data, is bascially the probability a galaxy is NOT detected in our survey as a function of gas-to-stellar mass ratio. Lifelines can also display the size of hte sample being considered at each x position (at-risk sample), the number of censored points, and the number of events.

In [ ]:
fig, ax = plt.subplots()
kmf.plot_survival_function(at_risk_counts=True, ax=ax)
ax.set_xlabel('gas-to-stellar mass ratio')
ax.set_ylabel('S')


You probably noticed that the survival curve does not start at 1, and the cumulative distribution function does not start at zero. This happens in left-censoring distributions when a lot of upper limits cluster at the low end. Conceptually, it means the data have values below the range of values where we have detections. If you think back to the idea of a medical study where we're tracking subjects' lifetimes, this is akin to some subjects dying before the study actually began. 

It's a bit weird, but it makes sense if you think about the nature of our data. We have a bunch of upper limits at the lower end, so the distribution is very unconstrained at low values.

### Your turn

Do the following:

- Take the catalog we've been using and select two different subsamples out of it. These can be anything you like (stellar mass regimes, color regimes, SFR)
- plot use the KaplanMeierFitter class to generate cumulative distribution functions of gas-to-stellar mass ratio for both. Put them on the same plot
- Get the mean gas-to-stellar mass ratio of each subset.

In [ ]:
# Your code here

## 3. Testing differences between survival functions using the log rank test

Next, we're going to use the logrank test, which is akin to the K-S test to determine if two distributions are signficantly different when there's no censoring. It compares survival function across the whole range of "lifetimes", i.e., the span of the data array. Depending on how your defined your two samples above, such a test may seem very trivial, but try it anyways.

Documentation for this test can be found at https://lifelines.readthedocs.io/en/latest/lifelines.statistics.html#lifelines.statistics.logrank_test

The basic call follows this syntax:
    results = logrank_test(array1, array1, detection_flag1, detection_flag2)

The output of this routine is a test statistic, and more importantly, the p-value. Like the K-S test, a p-value < 0.05 is generally considered significant.

***Important: The implementation of the log-rank test in lifelines assumes everything is right-censored (data are lower limits). So we have to flip the sign of the data first***

In [ ]:
# Your code here

## Final remarks

This tutorial only scratches the surface of what can be done with censored data, especially survival analysis statistical techniques. For tutorials that include how to test for correlations and fit lines to data sets with censoring, see https://github.com/karenlmasters/HaverfordGalaxyGroup/blob/main/BasicCode_Statistics/SurvivalAnalysisTutorial.ipynb

Other resources I've found useful:

- Modern Statistical Methods for Astronomy: With R Applications by Feigelson & Babu
- STATISTICS FOR CENSORED ENVIRONMENTAL DATA USING MINITAB AND R by Dennis Helsel
- the Astrostatistics summer school at PSU